In [ ]:
!pip install -q langchain langchain-community langchain-huggingface bitsandbytes accelerate sentence-transformers pypdf python-dotenv torch transformers qdrant-client langchain-qdrant fastembed

In [1]:
import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    pdf_file = "/content/drive/MyDrive/RAG_PIPELINE_01/test.pdf"
    if not os.path.exists(pdf_file):
        print("Please upload your PDF file:")
        uploaded = files.upload()
        if uploaded:
            uploaded_filename = list(uploaded.keys())[0]
            pdf_file = uploaded_filename
else:
    # Local Windows execution
    pdf_file = "RAG_PIPELINE/test.pdf" # Default local workspace path
    print(f"Running locally. Expected PDF path: {os.path.abspath(pdf_file)}")
    if not os.path.exists(pdf_file):
        print(f"Warning: PDF file not found at {pdf_file}. Please check the path.")

Running locally. Expected PDF path: d:\MODELS\LocalModel001\RAG_PIPELINE\test.pdf


In [2]:
import os
import sys
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    db_directory = "/content/drive/MyDrive/RAG_PIPELINE_01/Quadrant_DB_01"
else:
    db_directory = "Quadrant_DB_01" # Local database folder

print(f"Initializing Qdrant client at: {os.path.abspath(db_directory)}")
client = QdrantClient(path=db_directory)

_embedding_model = None

def get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = FastEmbedEmbeddings(
            model_name="BAAI/bge-base-en-v1.5"
        )
    return _embedding_model

def build_vector_db(pdf_path: str, _client):
    if not os.path.exists(pdf_path):
        print(f"Error: PDF file not found at {pdf_path}")
        return None

    emb_model = get_embedding_model()

    print(f"Loading document: {pdf_path}...")
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    print("Splitting document into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(docs)

    if chunks:
        collection_name = "Deep_Learning"
        # Check if the collection exists, and if not, create it first using the client
        if not _client.collection_exists(collection_name=collection_name):
            print(f"Creating collection '{collection_name}'...")
            # BAAI/bge-base-en-v1.5 has an embedding size of 768
            _client.create_collection(
                collection_name=collection_name,
                vectors_config=models.VectorParams(
                    size=768,
                    distance=models.Distance.COSINE
                )
            )
        
        # Instantiate QdrantVectorStore directly and use add_documents to support custom clients
        print(f"Storing {len(chunks)} chunks into vector DB at '{db_directory}'...")
        vectorstore = QdrantVectorStore(
            client=_client,
            embedding=emb_model,
            collection_name=collection_name
        )
        vectorstore.add_documents(documents=chunks)
        
        print("Chunks stored successfully!")
        return vectorstore
    else:
        print("No chunks created.")
        return None

C:\Users\26050050\AppData\Local\Temp\ipykernel_26264\1075341475.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\MODELS\LocalModel001\generativeAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing Qdrant client at: d:\MODELS\LocalModel001\Quadrant_DB_01


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch
from transformers import BitsAndBytesConfig

def get_llm():
    model_id = "microsoft/Phi-3-mini-4k-instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Check if GPU (CUDA) is available
    if torch.cuda.is_available():
        print("CUDA available! Loading model in 4-bit quantization...")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            quantization_config=quantization_config,
            device_map="auto"
        )
    else:
        print("CUDA not available. Loading model on CPU in float32 (this may be slow)...")
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
            device_map="cpu"
        )

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=300,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return HuggingFacePipeline(pipeline=pipe)

llm = get_llm()
embedding_model = get_embedding_model()

In [ ]:
import time
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

try:
    collections_response = client.get_collections()
    existing_collections = [col.name for col in collections_response.collections]
except Exception:
    existing_collections = []

if "Deep_Learning" not in existing_collections:
    print("Collection 'Deep_Learning' not found. Building Vector Database from PDF...")
    vectorstore = build_vector_db(pdf_file, client)
else:
    print("Collection 'Deep_Learning' found on disk. Loading...")
    # Load the database using the shared client
    vectorstore = QdrantVectorStore(
        client=client, 
        embedding=embedding_model, 
        collection_name="Deep_Learning"
    )

retriever_response = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.5}
)

print("\nRAG system initialized successfully!")
print("Type your query below. Press '0' to exit.\n")

def ask_rag(query_text: str):
    docs = retriever_response.invoke(query_text)
    context = "\n\n".join([doc.page_content for doc in docs])

    messages = [
        SystemMessage(
            content="You are a helpful AI assistant. Use only the provided context to answer the question. If the answer is not present in the context, say: 'I could not find the answer in the document.'"
        ),
        HumanMessage(
            content=f"Context:\n{context}\n\nQuestion: {query_text}\n\nRemember: Answer based ONLY on the context. If the answer is not present, say 'I could not find the answer in the document.'"
        )
    ]

    formatted_messages = [
        {"role": "system" if isinstance(msg, SystemMessage) else "user", "content": msg.content}
        for msg in messages
    ]
    
    tokenizer = llm.pipeline.tokenizer
    final_prompt = tokenizer.apply_chat_template(
        formatted_messages, tokenize=False, add_generation_prompt=True
    )

    start_time = time.perf_counter()
    response = llm.invoke(final_prompt)
    end_time = time.perf_counter()
    
    return response.strip(), end_time - start_time

# Interactive Loop
while True:
    try:
        query = input("You : ").strip()
        if query == "0":
            print("Goodbye!")
            break
        if not query:
            continue

        response, duration = ask_rag(query)
        print(f"\nAI : {response}")
        print(f"\nResponse Time: {duration:.2f} seconds\n")
    except (KeyboardInterrupt, EOFError):
        print("\nGoodbye!")
        break

Collection 'Deep_Learning' not found. Building Vector Database from PDF...
Loading document: RAG_PIPELINE/test.pdf...
Splitting document into chunks...
Creating collection 'Deep_Learning'...
Storing 2013 chunks into vector DB at 'Quadrant_DB_01'...
